# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Ляшко Леся'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 11       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 11   ·   Ляшко Леся   ·   ШІ-32

Ваші вісім стовпців:
   1. Телефон
   2. Примітка
   3. ЛН
   4. Виписка
   5. Ліжко днів
   6. Прокальцитонін
   7. Білок1
   8. Гематокрит2

Ваша пара для аналізу зв’язку: Вік та spO2 без дот

Ваше додаткове питання:
   Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,Примітка,ЛН,Виписка,Ліжко днів,Прокальцитонін,Білок1,Гематокрит2
0,0958469788 дружина Ганна,NaN,ІІІ-IV,2022-03-10 00:00:00,14,0.126,70.0,45.0
1,"0671778486, 0994887663 дружина Оксана","отримував актемра, часті носові кровотечі, втрата нюху",ІІ-ІІІ,2022-03-16 00:00:00,22,NaN,61.8,57.0
2,"0958768045, чоловік Степан 0951784726",часта втрата свідомосі в стаціонарі,ІІ,2022-01-18 00:00:00,20,NaN,80.5,37.0
3,0959819331. дочка Валентина 0960845205,панкреонекроз,ІІІ-IV,2022-01-13 00:00:00,20,NaN,62.6,34.0
4,"0952294866, дочка Дарина 0678360161",NaN,ІІ-ІІІ,2022-03-11 00:00:00,27,NaN,72.0,48.0
5,0959922983 сестра Софія,NaN,ІІ-ІІІ,2021-12-26 00:00:00,9(5 у реан),NaN,73.0,36.0
6,"0998073702, дружина Наталія 0998382226","глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс",ІІ-ІІІ,2022-05-11 00:00:00,18,NaN,76.0,44.5
7,"0637258771, син Сергій 0968413301",NaN,І,2022-05-28 00:00:00,10,NaN,88.0,42.0


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Телефон         120 non-null    object 
 1   Примітка        45 non-null     object 
 2   ЛН              121 non-null    object 
 3   Виписка         120 non-null    object 
 4   Ліжко днів      120 non-null    object 
 5   Прокальцитонін  46 non-null     float64
 6   Білок1          118 non-null    float64
 7   Гематокрит2     101 non-null    float64
dtypes: float64(3), object(5)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- Примітка
    непорожніх 45 з 121 | унікальних 44 | типи: str×45
    приклади: ["'отримував актемра, часті носові кровотечі, втрата нюху'", "'часта втрата свідомосі в стаціонарі'", "'панкреонекроз'", "'глюкозурія, кетонові тіла в сечі, ЦД, фібриляція передс'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- Ліжко днів
    непорожніх 120 з 121 | унікальних 40 | типи: int×119, str×1
    приклади: ['14', '22', '

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'Телефон': ('ідентифікатор', 'ідентифікують спосіб зв’язку з пацієнтом.'),
    'Примітка': ('номінальна', 'текстовий опис особливостей випадку без природного числового порядку між різними записами.'),
    'ЛН': ('порядкова', 'Рівні легеневої недостатності мають природний порядок тяжкості, але відстані між сусідніми рівнями не задані.'),
    'Виписка': ('інтервальна', 'Календарна дата має змістовні різниці в днях, але нуль календаря умовний і відношення дат не має змісту.'),
    'Ліжко днів': ('відношень', 'Кількість днів перебування має природний нуль, а відношення тривалостей, наприклад 20 до 10 днів, має зміст.'),
    'Прокальцитонін': ('відношень', 'Концентрація має природний фізичний нуль, тому осмислені і різниці, і відношення між значеннями.'),
    'Білок1': ('відношень', 'Рівень загального білка є кількісною концентрацією з природним нулем, тому відношення значень має зміст.'),
    'Гематокрит2': ('відношень', 'Гематокрит є кількісною часткою з природним нулем, тому можна порівнювати відношення значень.')
}

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,ідентифікують спосіб зв’язку з пацієнтом.
1,Примітка,номінальна,текстовий опис особливостей випадку без природного число...
2,ЛН,порядкова,Рівні легеневої недостатності мають природний порядок тя...
3,Виписка,інтервальна,"Календарна дата має змістовні різниці в днях, але нуль к..."
4,Ліжко днів,відношень,"Кількість днів перебування має природний нуль, а відноше..."
5,Прокальцитонін,відношень,"Концентрація має природний фізичний нуль, тому осмислені..."
6,Білок1,відношень,Рівень загального білка є кількісною концентрацією з при...
7,Гематокрит2,відношень,"Гематокрит є кількісною часткою з природним нулем, тому ..."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
def _numeric_values(стовпець):
    """Перетворює допустимі кількісні значення на числа."""
    s = df[стовпець]
    if стовпець == 'Ліжко днів':
        # Напр. "9(5 у реан)" -> загальна тривалість 9 днів.
        return pd.to_numeric(s.astype(str).str.extract(r'([0-9]+(?:[.,][0-9]+)?)')[0]
                             .str.replace(',', '.', regex=False), errors='coerce').dropna()
    return pd.to_numeric(s, errors='coerce').dropna()

def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р
    if len(з):
        р['мода'] = з.mode().iloc[0] if len(з.mode()) else None
        р['частота моди'] = int(з.value_counts().iloc[0])

    if ш == 'порядкова':
        if стовпець == 'ЛН':
            порядок = {'І': 1, 'І-ІІ': 1.5, 'ІІ': 2, 'ІІ-ІІІ': 2.5, 'ІІІ': 3, 'ІІІ-IV': 3.5}
            num = з.map(порядок).dropna()
        else:
            num = pd.to_numeric(з, errors='coerce').dropna()
        if len(num):
            р['медіана'] = float(num.median())
            р['Q1'] = float(num.quantile(.25))
            р['Q3'] = float(num.quantile(.75))

    elif ш == 'інтервальна':
        if стовпець == 'Виписка':
            d = pd.to_datetime(з, errors='coerce').dropna()
            origin = d.min()
            num = (d - origin).dt.total_seconds() / 86400
            р['середня дата'] = origin + pd.to_timedelta(num.mean(), unit='D')
            р['стандартне відхилення, днів'] = float(num.std())
        else:
            num = pd.to_numeric(з, errors='coerce').dropna()
            if len(num):
                р['середнє'] = float(num.mean())
                р['стандартне відхилення'] = float(num.std())

    elif ш in ('відношень', 'абсолютна'):
        num = _numeric_values(стовпець)
        if len(num):
            mean = num.mean()
            р['середнє'] = float(mean)
            р['стандартне відхилення'] = float(num.std())
            р['коефіцієнт варіації'] = float(num.std() / mean) if mean != 0 else np.nan

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон  (ідентифікатор)
    {'непорожніх': 120, 'унікальних': 120}

--- Примітка  (номінальна)
    {'непорожніх': 45, 'унікальних': 44, 'мода': 'Поліалергетик', 'частота моди': 2}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': 'ІІ-ІІІ', 'частота моди': 59, 'медіана': 2.5, 'Q1': 2.0, 'Q3': 2.5}

--- Виписка  (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': datetime.datetime(2021, 9, 7, 0, 0), 'частота моди': 3, 'середня дата': Timestamp('2022-01-29 18:06:06.101694915'), 'стандартне відхилення, днів': 109.52398452859599}

--- Ліжко днів  (відношень)
    {'непорожніх': 120, 'унікальних': 40, 'мода': 10, 'частота моди': 12, 'середнє': 17.883333333333333, 'стандартне відхилення': 12.936940355830204, 'коефіцієнт варіації': 0.7234076620221922}

--- Прокальцитонін  (відношень)
    {'непорожніх': 46, 'унікальних': 17, 'мода': np.float64(0.1), 'частота моди': 30, 'середнє': 0.7529565217391306, 'стандартне відхилення': 3.010374682892641, 'коефіцієнт варі

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Примітка,45,44,Поліалергетик,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-09-07 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ліжко днів,120.0,40.0,10.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,3.010375,0.043,0.1,0.1,0.1205,18.22
Білок1,118.0,NaN,NaN,NaN,66.75339,8.37849,50.0,60.0,66.0,72.0,88.0
Гематокрит2,101.0,NaN,NaN,NaN,41.140594,7.565992,16.0,36.0,40.0,46.0,57.0


In [10]:
телефон_коди = pd.Categorical(df['Телефон']).codes
примітка_коди = pd.Categorical(df['Примітка']).codes
лн_порядок = {'І': 1, 'І-ІІ': 1.5, 'ІІ': 2, 'ІІ-ІІІ': 2.5, 'ІІІ': 3, 'ІІІ-IV': 3.5}
лн_коди = df['ЛН'].map(лн_порядок)

БЕЗГЛУЗДІ = [
('Телефон', 'середнє кодів категорій', float(телефон_коди[телефон_коди >= 0].mean()),
 'Телефон є ідентифікатором: їх середнє не описує жодної властивості пацієнтів.'),
('Примітка', 'середнє кодів категорій', float(примітка_коди[примітка_коди >= 0].mean()),
 'примітки є номінальними категоріями без порядку, тому арифметика над довільними кодами категорій не має змісту.'),
('ЛН', 'середнє умовних рангів', float(лн_коди.mean()),
 'ЛН є порядковою шкалою: порядок рівнів відомий, але відстані між ними не гарантовано рівні, тому середнє умовних рангів некоректне.')
]


assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,Телефон,середнє кодів категорій,59.500000,Телефон є ідентифікатором: їх середнє не описує жодної в...
1,Примітка,середнє кодів категорій,21.200000,"примітки є номінальними категоріями без порядку, тому ар..."
2,ЛН,середнє умовних рангів,2.285124,"ЛН є порядковою шкалою: порядок рівнів відомий, але відс..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × spO2 без дот

Вік: шкала — не у вашому списку
spO2 без дот: шкала — не у вашому списку

   Вік  spO2 без дот
0   55          90.0
1   43          83.0
2   54          78.0
3   68          82.0
4   65          80.0
5   66          72.0


In [25]:
from scipy.stats import pearsonr

ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = ('Вік і spO2 без дотації є кількісними змінними шкали відношень. '
                   'Тому для оцінювання лінійного зв’язку між ними кореляція Пірсона є допустимою.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

пара = df[[A, B]].apply(pd.to_numeric, errors='coerce').dropna()
r, p_value = pearsonr(пара[A], пара[B])
print(ЧОМУ_САМЕ_ВОНА)
print(f'n = {len(пара)}, r = {r:.4f}')

if abs(r) < 0.3:
    сила = 'слабкий'
elif abs(r) < 0.7:
    сила = 'помірний'
else:
    сила = 'сильний'
напрям = 'прямий' if r > 0 else 'обернений'
print(f'Інтерпретація: {сила} {напрям} лінійний зв’язок.')


Вік і spO2 без дотації є кількісними змінними шкали відношень. Тому для оцінювання лінійного зв’язку між ними кореляція Пірсона є допустимою.
n = 120, r = -0.1463
Інтерпретація: слабкий обернений лінійний зв’язок.


In [26]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.
лн_коди_цілі = {'І': 1, 'І-ІІ': 2, 'ІІ': 3, 'ІІ-ІІІ': 4, 'ІІІ': 5, 'ІІІ-IV': 6}
контр = pd.DataFrame({
    'Вік': pd.to_numeric(df['Вік'], errors='coerce'),
    'ЛН_код': df['ЛН'].map(лн_коди_цілі)
}).dropna()

r_bad, p_bad = pearsonr(контр['Вік'], контр['ЛН_код'])
print(f'Некоректний Пірсона(Вік, умовний код ЛН) = {r_bad:.4f}')
print('Це число не слід інтерпретувати як кореляцію з ЛН, бо коди 1–6 задають '      'штучно рівні відстані між порядковими категоріями. Для порядкової змінної '      'доречніше використовувати ранговий підхід, наприклад Спірмена.')

Некоректний Пірсона(Вік, умовний код ЛН) = 0.3977
Це число не слід інтерпретувати як кореляцію з ЛН, бо коди 1–6 задають штучно рівні відстані між порядковими категоріями. Для порядкової змінної доречніше використовувати ранговий підхід, наприклад Спірмена.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'Телефон': 'Виключити з моделі: це ідентифікатор, а не предиктивна вимірювана ознака.',
    'Примітка': 'Для простої табличної моделі one-hot кодувати категорії; не вводити штучний порядок текстових значень.',
    'ЛН': 'Перетворити на впорядковану категорію із явно заданим медичним порядком рівнів тяжкості.',
    'Виписка': 'Перетворити дату на змістовну тривалість від опорної дати, після чого за потреби стандартизувати.',
    'Ліжко днів': 'Очистити текстові записи до числа днів і стандартизувати як шкалу відношень.',
    'Прокальцитонін': 'Як кількісну ознаку очистити пропуски; через сильну асиметрію можливе log1p-перетворення і стандартизація.',
    'Білок1': 'Як кількісну ознаку шкали відношень стандартизувати після обробки пропущених значень.',
    'Гематокрит2': 'Як кількісну ознаку шкали відношень стандартизувати після обробки пропущених значень.'
}

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"Виключити з моделі: це ідентифікатор, а не предиктивна в..."
1,Примітка,номінальна,Для простої табличної моделі one-hot кодувати категорії;...
2,ЛН,порядкова,Перетворити на впорядковану категорію із явно заданим ме...
3,Виписка,інтервальна,Перетворити дату на змістовну тривалість від опорної дат...
4,Ліжко днів,відношень,Очистити текстові записи до числа днів і стандартизувати...
5,Прокальцитонін,відношень,Як кількісну ознаку очистити пропуски; через сильну асим...
6,Білок1,відношень,Як кількісну ознаку шкали відношень стандартизувати післ...
7,Гематокрит2,відношень,Як кількісну ознаку шкали відношень стандартизувати післ...


In [15]:
# Реалізація підготовки для стовпців різних шкал.
prepared = pd.DataFrame(index=df.index)

порядок_лн = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']
лн_cat = pd.Categorical(df['ЛН'], categories=порядок_лн, ordered=True)
prepared['ЛН_ordinal'] = pd.Series(лн_cat.codes, index=df.index).replace(-1, np.nan)

виписка_dt = pd.to_datetime(df['Виписка'], errors='coerce')
prepared['Виписка_days_from_min'] = (виписка_dt - виписка_dt.min()).dt.days

білок = pd.to_numeric(df['Білок1'], errors='coerce')
prepared['Білок1_z'] = (білок - білок.mean()) / білок.std()

ліжко = pd.to_numeric(df['Ліжко днів'].astype(str)
                      .str.extract(r'([0-9]+(?:[.,][0-9]+)?)')[0]
                      .str.replace(',', '.', regex=False), errors='coerce')
prepared['Ліжко_днів_clean'] = ліжко

prepared.head(10)


,ЛН_ordinal,Виписка_days_from_min,Білок1_z,Ліжко_днів_clean
0,5,317.0,0.387493,14.0
1,3,323.0,-0.591203,22.0
2,2,266.0,1.640703,20.0
3,5,261.0,-0.495721,20.0
4,3,318.0,0.626200,27.0
5,3,243.0,0.745553,9.0
6,3,379.0,1.103613,18.0
7,0,396.0,2.535852,10.0
8,2,381.0,2.416499,17.0
9,4,206.0,0.626200,7.0


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [24]:
ВИПРАВЛЕННЯ = [
    (1, 'Для номінальної змінної «помер/жив» обчислено середнє арифметичне.',
     'Коди 1 і 2 є лише позначеннями двох категорій результату лікування, тому 1.50 не є станом «посередині»',
     'Потрібно подати відсоткове співвідношення, до прикалду: 61 пацієнт помер, 60 вижили; це приблизно 50.4% і 49.6%.'),
    (2, 'Різні текстові записи однієї й тієї самої статі помилково трактуються як окремі категорії.',
     'Значення «ж» і «жін» означають жіночу стать, а «ч» і «чол» — чоловічу.',
     'Привести до загального вигляду категорій. Після отримаємо 75 жінок і 46 чоловіків.'),
    (3, 'Порядкову змінну «ступінь КТ» відсортовано лексикографічно.',
     'Алфавітний порядок назв не відповідає порядку тяжкості, а середнє припускає однакові відстані між рівнями.',
     'задати явний порядок легкий < середній < важкий < вкрай важкий < критичний; використовувати частоти, медіану.'),
    (4, 'Стовпець «КТ» може містити діапазон величини, але код бере лише ті значення, які вже є числами.',
     'Числа на кшталт 0.8 означають частку 80%, тоді як рядки «20-30%» або «до 17%» відкидаються через "coerce", тому середнє 39.6% обчислене не на всьому датасеті',
     'Перетворити всі записи до єдиного вигляду; для інтервалів узяти, наприклад, середину'),
    (5, 'У «spO2 без дот» змішані відсотки та частки, але середнє обчислено без приведення до одного вигляду.',
     'Значення 0.9 відповідає 90%, але в середньому враховується як 0.9, занижуючи результат 82.05%.',
     'Значення в інтервалі 0–1 перевести у відсотки множенням на 100.'),
    (6, 'До календарних дат застосовано коефіцієнт варіації, який потребує змістовного природного нуля.',
     'Числове значення дати залежить від довільно вибраного вигляду, тому середнє і CV зміняться при зміні початку відліку та не характеризують однорідність госпіталізацій.',
     'Для дат аналізувати інтервали або розкид у днях від змістовної опорної дати; наприклад, у наборі госпіталізації охоплюють 463 дні.'),
    (7, 'Для двох номінальних змінних застосовано кореляцію Пірсона після довільного кодування статі числами 0/1.',
     'Числові коди категорій не утворюють кількісної шкали, тому кореляція не є належною мірою зв’язку і висновок про його відсутність некоректний.',
     'Побудувати таблицю спряженості, виконати χ²-тест і обчислити V Крамера'),
    (8, 'Мін-макс механічно застосовано до всіх числових стовпців без перевірки шкали та варіативності.',
     'Числовий dtype не гарантує кількісної шкали; крім того, сталий стовпець «ФЕРИТ2» має max = min, тому знаменник дорівнює нулю і утворюються NaN.',
     'Нормалізувати лише доречні кількісні ознаки та перед цим перевіряти max > min; коди категорій не нормалізувати, сталі ознаки вилучати.'),
    (9, 'Використано df.dropna() для всього набору без аналізу змісту пропусків.',
     'У 77 стовпцях пропуски дуже поширені, тому повне видалення рядків залишає 0 пацієнтів; крім того, деякі порожні клітинки можуть мати зміст.',
     'Обробляти пропуски окремо для кожної ознаки, оскільки вони можуть бути змістовними'),
    (10, '«Дата розтину» використана як дата летальності, хоча вона виникає після смерті пацієнта.',
     'Наявність дати розтину означає летальний результат, тому r ≈ 1 є витоком цільової інформації, а не корисною прогностичною залежністю.',
     'Виключити такі ознаки та оцінювати модель лише на даних, доступних на момент прогнозу; важливість визначати після коректного train/test поділу.')
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])


,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,Для номінальної змінної «помер/жив» обчислено середнє ар...,Коди 1 і 2 є лише позначеннями двох категорій результату...,"Потрібно подати відсоткове співвідношення, до прикалду: ..."
1,2,Різні текстові записи однієї й тієї самої статі помилков...,"Значення «ж» і «жін» означають жіночу стать, а «ч» і «чо...",Привести до загального вигляду категорій. Після отримаєм...
2,3,Порядкову змінну «ступінь КТ» відсортовано лексикографічно.,"Алфавітний порядок назв не відповідає порядку тяжкості, ...",задати явний порядок легкий < середній < важкий < вкрай ...
3,4,"Стовпець «КТ» може містити діапазон величини, але код бе...","Числа на кшталт 0.8 означають частку 80%, тоді як рядки ...",Перетворити всі записи до єдиного вигляду; для інтервалі...
4,5,"У «spO2 без дот» змішані відсотки та частки, але середнє...","Значення 0.9 відповідає 90%, але в середньому враховуєть...",Значення в інтервалі 0–1 перевести у відсотки множенням ...
5,6,"До календарних дат застосовано коефіцієнт варіації, який...",Числове значення дати залежить від довільно вибраного ви...,Для дат аналізувати інтервали або розкид у днях від зміс...
6,7,Для двох номінальних змінних застосовано кореляцію Пірсо...,"Числові коди категорій не утворюють кількісної шкали, то...","Побудувати таблицю спряженості, виконати χ²-тест і обчис..."
7,8,Мін-макс механічно застосовано до всіх числових стовпців...,"Числовий dtype не гарантує кількісної шкали; крім того, ...",Нормалізувати лише доречні кількісні ознаки та перед цим...
8,9,Використано df.dropna() для всього набору без аналізу зм...,"У 77 стовпцях пропуски дуже поширені, тому повне видален...","Обробляти пропуски окремо для кожної ознаки, оскільки во..."
9,10,"«Дата розтину» використана як дата летальності, хоча вон...","Наявність дати розтину означає летальний результат, тому...","Виключити такі ознаки та оцінювати модель лише на даних,..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:

результат = df['помер/жив'].value_counts().sort_index()
print('1) Результат лікування')
print(pd.DataFrame({'кількість': результат, 'частка, %': (результат / len(df) * 100).round(1)}))
print()

середній_результат = df['помер/жив'].mean()
print('Середній результат лікування:', round(середній_результат, 2))
print()




стать_clean = df['Стать'].astype(str).str.strip().replace({'жін': 'ж', 'чол': 'ч'})
print('\n2) Стать після уніфікації')
print(стать_clean.value_counts())

стать = df['Стать'].value_counts()
print('У вибірці представлено', df['Стать'].nunique(), 'статі:')
print(стать.to_string())
print()



spo2_raw = pd.to_numeric(df['spO2 без дот'], errors='coerce')
spo2_clean = spo2_raw.where(spo2_raw > 1, spo2_raw * 100)
print('\n3) spO2 без дотації')
print('Неправильне середнє:', round(spo2_raw.mean(), 2))
print('Правильне середнє:', round(spo2_clean.mean(), 2))




1) Результат лікування
           кількість  частка, %
помер/жив                      
1                 61       50.4
2                 60       49.6

Середній результат лікування: 1.5


2) Стать після уніфікації
Стать
ж    75
ч    46
Name: count, dtype: int64
У вибірці представлено 4 статі:
Стать
ж      56
ч      34
жін    19
чол    12


3) spO2 без дотації
Неправильне середнє: 82.05
Правильне середнє: 82.79


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'У датасеті результат лікування є номінальною бінарною змінною: 61 пацієнт має значення 1, а 60 — значення 2, тому змістовно порівнювати частоти та частки, а не середнє значення кодів.',
    'Після приведення spO2 без дотації до єдиного вигляду середнє становить приблизно 82.79%, а зв’язок між віком і spO2 є слабким оберненим: приблизно -0.271.',
    'Перевірка помилкового аналізу показала, що тип pandas сам по собі не визначає допустиму статистику: ознаки, зо приведені не до єдиного вигляду, потребують окремої змістовної перевірки.'
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У датасеті результат лікування є номінальною бінарною змінною: 61 пацієнт має значення 1, а 60 — значення 2, тому змістовно порівнювати частоти та частки, а не середнє значення кодів.
2. Після приведення spO2 без дотації до єдиного вигляду середнє становить приблизно 82.79%, а зв’язок між віком і spO2 є слабким оберненим: приблизно -0.271.
3. Перевірка помилкового аналізу показала, що тип pandas сам по собі не визначає допустиму статистику: ознаки, зо приведені не до єдиного вигляду, потребують окремої змістовної перевірки.


In [21]:
#Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = 'До перетворення «Вік» є кількісною змінною шкали відношень: вік має природний нуль, різниці між значеннями мають зміст, і твердження «60 років удвічі більше за 30 років» математично коректне щодо тривалості життя. Після поділу на групи «до 40», «40–60», «понад 60» змінна стає порядковою категоріальною: категорії мають природний порядок, але відстань між ними вже не визначена. Ми втрачаємо точне значення віку та частину інформації про відмінності всередині груп: наприклад, 41 і 60 років стають однією категорією. Натомість отримуємо простіше представлення, яке легше використовувати для групових порівнянь. 3 пацієнти «до 40», 37 — «40–60» і 81 — «понад 60», тому після групування особливо помітною стає перевага старшої вікової групи'
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?

До перетворення «Вік» є кількісною змінною шкали відношень: вік має природний нуль, різниці між значеннями мають зміст, і твердження «60 років удвічі більше за 30 років» математично коректне щодо тривалості життя. Після поділу на групи «до 40», «40–60», «понад 60» змінна стає порядковою категоріальною: категорії мають природний порядок, але відстань між ними вже не визначена. Ми втрачаємо точне значення віку та частину інформації про відмінності всередині груп: наприклад, 41 і 60 років стають однією категорією. Натомість отримуємо простіше представлення, яке легше використовувати для групових порівнянь. 3 пацієнти «до 40», 37 — «40–60» і 81 — «понад 60», тому після групування особливо помітною стає перевага старшої вікової групи


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'синтаксис',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'синтаксис',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = 'швидше опрацьовувала дані, перевірила, чи правильно зрозуміла, що саме це є неправильним опрацюванням даних? привела до кращого вигляду'
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 11 готова до захисту.
Ляшко Леся, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.